# Part 3 - NLP and Sequence Modeling

In this notebook I am going to work on customer support text data and try to classify it by sentiment. The classes are positive, neutral and negative.

## Importing libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

## Task 1 - Dataset Understanding

First I will load the dataset and look at what is inside it.

In [ ]:
df = pd.read_csv('customer_support_text_classification.csv')
df.head()

In [ ]:
print('Number of records:', len(df))
print()
print('Columns:', df.columns.tolist())

In [ ]:
print('Target labels and their counts:')
print(df['sentiment_label'].value_counts())

In [ ]:
# average word count
print('Average text length (words):', df['word_count'].mean())
print('Min:', df['word_count'].min())
print('Max:', df['word_count'].max())

In [ ]:
print('Sample text records:')
print()
for i in range(5):
    print(f"[{i}] Label: {df['sentiment_label'].iloc[i]}")
    print(f"     Text : {df['customer_message'].iloc[i]}")
    print()

In [ ]:
# class distribution plot
counts = df['sentiment_label'].value_counts()
plt.bar(counts.index, counts.values, color=['gray', 'red', 'green'])
plt.title('Class Distribution')
plt.show()

The dataset has 1500 rows and 3 classes. The classes look fairly balanced which is good. Average message length is around 12-13 words.

## Task 2 - Text Preprocessing

Now I will clean the text. This means:
- converting everything to lowercase
- removing special characters and numbers
- tokenizing
- removing stopwords

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:
def preprocess(text):
    text = str(text).lower()
    # remove anything that is not a letter or space
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

In [ ]:
df['clean_text'] = df['customer_message'].apply(preprocess)

# check what it looks like
print('Before:', df['customer_message'].iloc[0])
print('After :', df['clean_text'].iloc[0])

In [ ]:
# check if any clean_text is empty after preprocessing
empty = df[df['clean_text'] == '']
print('Empty rows after cleaning:', len(empty))

In [ ]:
# I am also checking the average length after cleaning just to compare
df['clean_word_count'] = df['clean_text'].apply(lambda x: len(x.split()))
print('Average clean word count:', df['clean_word_count'].mean())

# also checking the average again from original column
print('Average original word count:', df['word_count'].mean())

## Task 3 - Text Vectorization

Machine learning models can not understand raw text. They only understand numbers. So we have to convert our words into numbers.

There are different ways to do this:
- **Bag of Words** - just counts how many times each word appears
- **TF-IDF** - similar but gives less importance to words that appear in almost every sentence (like 'the' or 'is')
- **Word Embeddings** - maps words to dense vectors based on meaning
- **Tokenizer sequences** - converts words to integer ids, used in deep learning

I will use both BoW and TF-IDF here to see the difference.

In [ ]:
# Bag of Words
bow_vectorizer = CountVectorizer(max_features=300)
X_bow = bow_vectorizer.fit_transform(df['clean_text'])

print('BoW matrix shape:', X_bow.shape)

In [ ]:
# TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=300)
X_tfidf = tfidf_vectorizer.fit_transform(df['clean_text'])

print('TF-IDF matrix shape:', X_tfidf.shape)

In [ ]:
# let's look at some of the vocabulary words found
vocab = tfidf_vectorizer.get_feature_names_out()
print('Sample vocabulary words:')
print(vocab[:30])

The text needs to be converted to vectors because:
- Models work with numbers, not strings
- Math operations (like dot products, gradients) only work on numeric data
- Vectorization also helps capture which words appear and how often

TF-IDF is usually better than plain BoW because it reduces the weight of very common words that don't really carry meaning.

## Task 4 - Baseline Model

I will use Logistic Regression with TF-IDF as the baseline model. It is simple and works well for text classification.

In [ ]:
# label encoding manually
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment_label'].map(label_map)

df['label'].value_counts()

In [ ]:
X = X_tfidf
y = df['label']

# splitting with test size 0.15 (15%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15)

In [ ]:
print('Train size:', X_train.shape[0])
print('Test size:', X_test.shape[0])

In [ ]:
lr_model = LogisticRegression(max_iter=200)
lr_model.fit(X_train, y_train)

In [ ]:
y_pred = lr_model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=['negative', 'neutral', 'positive']))

In [ ]:
# confusion matrix and class distribution side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['sentiment_label'].value_counts()
axes[0].bar(counts.index, counts.values, color=['red', 'gray', 'green'])
axes[0].set_title('Class Distribution')

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['negative', 'neutral', 'positive'])
disp.plot(ax=axes[1])
axes[1].set_title('Confusion Matrix - Logistic Regression')

plt.tight_layout()
plt.savefig('results/model_evaluation.png', dpi=100)
plt.show()

In [ ]:
# saving the evaluation report to csv
report = classification_report(y_test, y_pred, target_names=['negative', 'neutral', 'positive'], output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv('results/model_evaluation.csv')
print('Saved.')

In [ ]:
# saving sample predictions
label_inv = {v: k for k, v in label_map.items()}
test_indices = y_test.index.tolist()

with open('results/sample_predictions.txt', 'w') as f:
    f.write('Sample Predictions from Logistic Regression + TF-IDF\n')
    f.write('=' * 55 + '\n\n')
    for i in range(15):
        idx = test_indices[i]
        actual = df['sentiment_label'].iloc[idx]
        pred = label_inv[y_pred[i]]
        msg = df['customer_message'].iloc[idx]
        match = 'CORRECT' if actual == pred else 'WRONG'
        f.write(f'[{i+1}] {match}\n')
        f.write(f'Message   : {msg}\n')
        f.write(f'Actual    : {actual}\n')
        f.write(f'Predicted : {pred}\n\n')

print('Saved predictions.')

The accuracy is really high - close to 100%. I think this is because the dataset is synthetic and the words in each class are probably quite predictable. In a real world case the model would not do this well.

## Task 5 - Sequence Model (LSTM Architecture)

For sequence modeling I will set up an LSTM. I was not able to fully train it because it takes too long without a GPU, so I am showing the architecture and explaining how it would work.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [ ]:
MAX_VOCAB = 5000
MAX_LEN = 30  # most messages are short so 30 should be okay

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(df['clean_text'])

sequences = tokenizer.texts_to_sequences(df['clean_text'])
padded = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

print('Shape of padded sequences:', padded.shape)

In [ ]:
# train test split for sequence model
y_seq = df['label'].values
X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(padded, y_seq, test_size=0.2)

In [ ]:
# building the LSTM model
lstm_model = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=64, input_length=MAX_LEN),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
lstm_model.summary()

### LSTM Architecture Explanation

Here is what each layer does:

**Input sequence**
Each customer message is converted to a sequence of integers using the tokenizer. For example "payment issue urgent" becomes [45, 12, 89]. These sequences are padded to length 30.

**Embedding Layer**
This converts each integer (word id) into a dense vector of size 64. So each word gets its own 64-dimensional representation. The embedding weights are learned during training.

**LSTM Layer**
This processes the sequence one word at a time and keeps a hidden state that carries information from earlier words. This is important because meaning sometimes depends on word order - "not happy" is very different from "happy".

**Output Layer**
A Dense layer with 3 neurons and softmax activation. It gives a probability for each of the 3 classes. The class with highest probability is the prediction.

**Loss Function**
We are using `sparse_categorical_crossentropy` because our labels are integers (0, 1, 2) not one-hot encoded.

**Evaluation Metric**
Accuracy and also classification report (precision, recall, f1) per class.

In [ ]:
# I am training just for 2 epochs to show it works, not for full training
history = lstm_model.fit(
    X_train_seq, y_train_seq,
    epochs=2,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

In [ ]:
loss, acc = lstm_model.evaluate(X_test_seq, y_test_seq)
print('LSTM Test Accuracy (2 epochs only):', round(acc, 4))

The LSTM even with only 2 epochs is doing quite well. With more epochs and tuning it would probably match or beat the logistic regression.

## Task 6 - Attention and Transformer Reflection

### Why RNNs struggle with long-term dependencies

RNNs process text one word at a time and pass a hidden state forward. The problem is that as the sequence gets longer, the information from the beginning of the sentence gets mixed in and diluted through so many steps. By the time you reach the end of a long sentence, the hidden state barely remembers what happened at the start. This is called the vanishing gradient problem - the gradients that are used to update the early weights become very small and basically stop learning.

For example in the sentence "The customer who called last Tuesday about the broken product is very angry", by the time you get to "very angry" a basic RNN might have mostly forgotten about "broken product".

### How LSTMs help

LSTMs solve this by introducing a cell state which is like a long-term memory that runs alongside the hidden state. There are three gates:
- **Forget gate** - decides what to throw away from memory
- **Input gate** - decides what new information to store
- **Output gate** - decides what to send to the next step

These gates let the model selectively remember or forget things. So important information from earlier in the sequence can survive for many steps without getting lost.

### What Attention solves

Even LSTMs struggle with very long sequences. Attention is a mechanism that lets the model look at any part of the input directly when producing an output, instead of relying on a single hidden state summary. In a sequence to sequence task like translation, when translating a word, attention lets the model focus on the specific input words that are most relevant to that output word.

This is much more flexible than just passing a single context vector.

### Why Transformers are important

Transformers use attention as their main building block and don't use recurrence at all. This means they can process all positions in the sequence at the same time (in parallel), which makes them much faster to train than RNNs and LSTMs.

Transformers are the reason models like BERT and GPT exist. They can learn very rich representations of language and can be pre-trained on huge amounts of text and then fine-tuned for specific tasks. This has basically made them the standard architecture for almost all NLP tasks today.

In generative AI, transformers are what makes it possible for models to generate long, coherent text because they can keep track of long-range context across the entire sequence without the memory problems of RNNs.